# Lesson 000.20 -- Playing The Symbol Game: Candidate Moves, Host Referees, and Monotonic Non-Increase

AuthorityEffect: None.

> **Claim-typing legend**
> - **[Pinned]**: reproduced from verified checked-in Symbol Game specifications (`l1_symbol_game_governor.py`).
> - **[Executed]**: verified by deterministic Python execution in this notebook.
> - **[Declared]**: pedagogical convention or beginner framing.
> - **[Open]**: questions left for future exploration.

## 1. The Big Idea: Two-Player Protocol and Host Referee

[Pinned] The Symbol Game is structured as a two-player retraction protocol between:
1. **The Proposer:** Proposes a candidate transition or state representation $c'$.
2. **The Host Referee:** Inspects the candidate move, recomputes topological invariants from scratch, and emits a binding verdict.

Crucially, the Proposer's self-reported claim has `AuthorityEffect: None`. The Host referee executes independent verification over exact rational arithmetic ($\mathbb{Q}$).

## 2. Monotonic Defect Non-Increase and Operation Typing

[Pinned] State transitions are governed by explicit operation types and admission rules:
1. **Syntax, Domain, and Carrier Bounds:** Coordinates must be exact rationals ($\mathbb{Q}$). Floats, NaNs, and booleans are strictly refused: `REFUSED_DOMAIN`. Carrier topology and cycle must match the declared graph: `REFUSED_CARRIER`.
2. **Section CUT ($s \to s'$, or shift $c' = c - ds$):** Adjusts vertex potential section elevations. Circulation and gauge class are invariant. A move is admitted iff circulation is preserved and section defect $D_c(s) = \sum |c_e - ds|$ does not increase ($D_{\text{new}} \le D_{\text{old}}$). If circulation alters, verdict is `REFUSED_COHOMOLOGY_BREACH`. If defect increases, verdict is `REFUSED_DEFECT_INCREASE`.
3. **Relational Repair ($c \to c'$):** Modifies edge claims. Admitted iff minimal obstruction does not increase: $\Phi_1(c') \le \Phi_1(c)$.
4. **Monotonicity vs. Finite Termination:** Monotonic non-increase guarantees that defect never worsens; finite termination on continuous rationals requires a quantized discrete state space, step quota, or well-founded rank.

In [ ]:
from fractions import Fraction as Q


def _parse_exact_q(val, name: str) -> Q:
    if isinstance(val, bool):
        raise TypeError(f"{name} must be exact int or Fraction, got bool")
    if isinstance(val, float):
        raise TypeError(f"{name} must be exact int or Fraction, got float {val}")
    try:
        return Q(val)
    except Exception as e:
        raise TypeError(f"{name} cannot be converted to Fraction: {e}")


def evaluate_move(c_old, c_new, edges, cycle, move_type="CUT"):
    # 1. Enforce declared fixed carrier and cycle
    EXPECTED_EDGES = ((0, 1), (1, 2), (2, 0))
    EXPECTED_CYCLE = {(0, 1): 1, (1, 2): 1, (2, 0): 1}
    if tuple(edges) != EXPECTED_EDGES or cycle != EXPECTED_CYCLE:
        return {
            "admitted": False,
            "verdict": "REFUSED_CARRIER",
            "reason": "unsupported carrier or cycle",
        }

    # 2. Check dimensions
    if len(c_old) != len(edges) or len(c_new) != len(edges):
        return {"admitted": False, "verdict": "REFUSED_SYNTAX", "reason": "dimension mismatch"}

    # 3. Validate exact rational coordinates (reject float, bool, NaN)
    try:
        c_old_q = tuple(_parse_exact_q(x, "c_old") for x in c_old)
        c_new_q = tuple(_parse_exact_q(x, "c_new") for x in c_new)
    except TypeError as e:
        return {"admitted": False, "verdict": "REFUSED_DOMAIN", "reason": str(e)}

    # 4. Host recomputes exact circulations independently
    circ_old = sum(val * cycle[e] for e, val in zip(edges, c_old_q))
    circ_new = sum(val * cycle[e] for e, val in zip(edges, c_new_q))

    phi1_old = abs(circ_old)
    phi1_new = abs(circ_new)

    if move_type == "CUT":
        if circ_new != circ_old:
            return {
                "admitted": False,
                "verdict": "REFUSED_COHOMOLOGY_BREACH",
                "reason": "Section CUT cannot alter circulation or gauge class",
                "circ_old": circ_old,
                "circ_new": circ_new,
            }

        # In Lesson 17 residual convention, representative mass equals section defect D_c(s)
        d_old = sum(abs(x) for x in c_old_q)
        d_new = sum(abs(x) for x in c_new_q)

        if d_new > d_old:
            return {
                "admitted": False,
                "verdict": "REFUSED_DEFECT_INCREASE",
                "d_old": d_old,
                "d_new": d_new,
                "phi1": phi1_old,
            }

        status = "MonotonicDescent" if d_new < d_old else "IdempotentStationary"
        return {
            "admitted": True,
            "verdict": "ADMITTED",
            "status": status,
            "d_old": d_old,
            "d_new": d_new,
            "phi1": phi1_old,
            "mass_old": d_old,
            "mass_new": d_new,
        }

    elif move_type == "RELATIONAL_REPAIR":
        if phi1_new > phi1_old:
            return {
                "admitted": False,
                "verdict": "REFUSED_DEFECT_INCREASE",
                "phi1_old": phi1_old,
                "phi1_new": phi1_new,
            }
        status = "MonotonicDescent" if phi1_new < phi1_old else "IdempotentStationary"
        return {
            "admitted": True,
            "verdict": "ADMITTED",
            "status": status,
            "phi1_old": phi1_old,
            "phi1_new": phi1_new,
            "mass_old": sum(abs(x) for x in c_old_q),
            "mass_new": sum(abs(x) for x in c_new_q),
        }
    else:
        return {
            "admitted": False,
            "verdict": "REFUSED_UNKNOWN_OPCODE",
            "reason": f"unknown move_type {move_type}",
        }


edges = ((0, 1), (1, 2), (2, 0))
cycle = {(0, 1): 1, (1, 2): 1, (2, 0): 1}

# Initial triangle state c = (2, -1, 0), D_c(s_0) = 3
c_init = tuple(map(Q, (2, -1, 0)))

# Test Candidate 1: Valid elevator cut shifting by s_1 = (0, 1, 0)
# Produces residual c' = (1, 0, 0) with D_c(s_1) = 1
c_valid = tuple(map(Q, (1, 0, 0)))
res1 = evaluate_move(c_init, c_valid, edges, cycle, move_type="CUT")

assert res1["admitted"] and res1["verdict"] == "ADMITTED"
assert res1["d_new"] == Q(1) < res1["d_old"]
assert res1["phi1"] == Q(1)

print("Candidate 1 Admitted (CUT):", res1)

## 2. Refusal Falsifiers: Defect Increase, Cohomology Breach, and Domain Breaches

[Pinned] The referee enforces typed refusals across multiple distinct failure modes:
1. **Defect Increase under CUT:** A move that increases section defect $D_c(s)$ (e.g. $(2, -1, 0) \to (3, -2, 0)$) is refused (`REFUSED_DEFECT_INCREASE`).
2. **Cohomology Breach under CUT:** A CUT cannot alter circulation. Proposing $(0, 0, 0)$ under `CUT` is refused (`REFUSED_COHOMOLOGY_BREACH`).
3. **Defect Increase under Relational Repair:** Proposing $(2, 2, 2)$ where $\Phi_1$ increases from $1 \to 6$ is refused (`REFUSED_DEFECT_INCREASE`).
4. **Exact Domain & Carrier Fences:** Floats and NaNs are refused (`REFUSED_DOMAIN`); invalid edge sets are refused (`REFUSED_CARRIER`).

In [ ]:
# Test Candidate 2: Bad elevator move that increases section defect (CUT)
c_worse = tuple(map(Q, (3, -2, 0)))
res2 = evaluate_move(c_init, c_worse, edges, cycle, move_type="CUT")
assert not res2["admitted"] and res2["verdict"] == "REFUSED_DEFECT_INCREASE"
assert res2["d_new"] == Q(5) > res2["d_old"]

# Test Candidate 3: Class-altering move under CUT (circulation changed 1 -> 0)
c_zero = tuple(map(Q, (0, 0, 0)))
res3 = evaluate_move(c_init, c_zero, edges, cycle, move_type="CUT")
assert not res3["admitted"] and res3["verdict"] == "REFUSED_COHOMOLOGY_BREACH"

# Test Candidate 4: Permitted relational repair admitting (0, 0, 0)
res4 = evaluate_move(c_init, c_zero, edges, cycle, move_type="RELATIONAL_REPAIR")
assert res4["admitted"] and res4["verdict"] == "ADMITTED"
assert res4["phi1_new"] == Q(0) < res4["phi1_old"]

# Test Candidate 5: Defect increase under relational repair ((2, 2, 2) has Phi_1 = 6)
c_bad = tuple(map(Q, (2, 2, 2)))
res5 = evaluate_move(c_init, c_bad, edges, cycle, move_type="RELATIONAL_REPAIR")
assert not res5["admitted"] and res5["verdict"] == "REFUSED_DEFECT_INCREASE"
assert res5["phi1_new"] == Q(6) > res5["phi1_old"]

# Test Candidate 6: Malformed syntax (length mismatch)
res6 = evaluate_move(c_init, (Q(1), Q(2)), edges, cycle)
assert not res6["admitted"] and res6["verdict"] == "REFUSED_SYNTAX"

# Test Candidate 7: Float input refused
res7 = evaluate_move((1.0, 0.0, 0.0), c_valid, edges, cycle)
assert not res7["admitted"] and res7["verdict"] == "REFUSED_DOMAIN"

# Test Candidate 8: NaN input refused
res8 = evaluate_move((float("nan"), 0.0, 0.0), c_valid, edges, cycle)
assert not res8["admitted"] and res8["verdict"] == "REFUSED_DOMAIN"

# Test Candidate 9: Unsupported carrier refused
res9 = evaluate_move(c_init, c_valid, ((0, 1), (1, 2)), cycle)
assert not res9["admitted"] and res9["verdict"] == "REFUSED_CARRIER"

# Test Candidate 10: Untrusted proposer self-reported AuthorityEffect is quarantined
untrusted_proposer_claim = {"verdict": "ADMITTED", "authority_effect": "Full"}
assert res2["verdict"] != untrusted_proposer_claim["verdict"]

print("All 10 referee refusal oracles and claim quarantine verified successfully!")

## 3. Diagnostic Probe (2026 Micro-Attention Check)

**Question:** Under the Symbol Game governor rules, why does the Host referee recompute $\Phi_1(c')$ and section defect $D_c(s)$ instead of accepting the Proposer's reported values?

- **A:** Recomputing is faster than parsing reported text.
- **B:** Proposer outputs have `AuthorityEffect: None` and are untrusted proposals; only host recomputation over exact rational arithmetic prevents uncertified defect increases.
- **C:** The Host referee needs floating point rounding.

*(Correct answer: **B**)*


In [ ]:
def diagnostic_probe(choice: str) -> bool:
    return choice.strip().upper() == "B"


assert diagnostic_probe("B")
assert not diagnostic_probe("A")
assert not diagnostic_probe("C")

# Student Exercise: From perturbed state c = (3, -2, 0) with D=5, verify that
# moving to optimal state c' = (1, 0, 0) with D=1 is admitted as a valid section CUT
c_perturb = tuple(map(Q, (3, -2, 0)))
c_opt = tuple(map(Q, (1, 0, 0)))

res_opt = evaluate_move(c_perturb, c_opt, edges, cycle, move_type="CUT")
assert res_opt["admitted"] and res_opt["verdict"] == "ADMITTED"
assert res_opt["d_new"] == Q(1) < res_opt["d_old"]  # Dropped from 5 to 1!

# Conversely, moving from optimal (1, 0, 0) to (3, -2, 0) is refused as DEFECT_INCREASE
res_rev = evaluate_move(c_opt, c_perturb, edges, cycle, move_type="CUT")
assert not res_rev["admitted"] and res_rev["verdict"] == "REFUSED_DEFECT_INCREASE"

print("Lesson 000.20 diagnostic probe and exercise verified successfully!")